# Evaluation: Fine-Tuned Model vs Context7 RAG Baseline

Compares two approaches on the held-out test set (100 examples):
1. **Fine-tuned** — QLoRA adapter on base model
2. **RAG (Context7 local docs)** — base model + retrieved doc chunks as context

Metrics: BLEU, ROUGE-L, CodeBLEU (identifier match), exact-match rate, and human-readable side-by-side.

## 1. Setup

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes rouge-score nltk sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = 'nmnhut-it/fine-tune-cocoos'
if not os.path.exists('fine-tune-cocoos'):
    !git clone https://github.com/{REPO}.git
os.chdir('fine-tune-cocoos')

## 2. Load Test Set

In [ ]:
from datasets import load_dataset

test_ds = load_dataset('json', data_files='data/test.jsonl', split='train')
print(f'Test examples: {len(test_ds)}')

## 3. Build Context7 RAG Index

Chunk the local docs, embed them, and build a vector index for retrieval.

In [ ]:
import glob
import re
import numpy as np
from sentence_transformers import SentenceTransformer

# Load and chunk all doc files
CHUNK_SIZE = 800  # chars per chunk
CHUNK_OVERLAP = 200

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping chunks at paragraph boundaries."""
    paragraphs = re.split(r'\n{2,}', text)
    chunks = []
    current = ''
    for para in paragraphs:
        if len(current) + len(para) > chunk_size and current:
            chunks.append(current.strip())
            # Keep overlap from end of current chunk
            current = current[-overlap:] + '\n\n' + para
        else:
            current = current + '\n\n' + para if current else para
    if current.strip():
        chunks.append(current.strip())
    return chunks

doc_chunks = []
doc_sources = []
for fpath in sorted(glob.glob('local-context7-cocos2d-x-only/docs/*.md')):
    with open(fpath) as f:
        text = f.read()
    fname = os.path.basename(fpath)
    chunks = chunk_text(text)
    doc_chunks.extend(chunks)
    doc_sources.extend([fname] * len(chunks))

print(f'Total chunks: {len(doc_chunks)}')
print(f'Avg chunk length: {np.mean([len(c) for c in doc_chunks]):.0f} chars')

# Embed chunks
embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embedder.encode(doc_chunks, show_progress_bar=True, convert_to_numpy=True)
print(f'Embeddings shape: {chunk_embeddings.shape}')

In [ ]:
def retrieve_context(query, top_k=5):
    """Retrieve top-k doc chunks most relevant to the query."""
    q_emb = embedder.encode([query], convert_to_numpy=True)
    scores = np.dot(chunk_embeddings, q_emb.T).squeeze()
    top_indices = np.argsort(scores)[-top_k:][::-1]
    retrieved = []
    for idx in top_indices:
        retrieved.append({
            'text': doc_chunks[idx],
            'source': doc_sources[idx],
            'score': float(scores[idx]),
        })
    return retrieved

# Quick test
results = retrieve_context('How to create a button in ccui?')
for r in results[:2]:
    print(f"[{r['source']}] score={r['score']:.3f}")
    print(r['text'][:150], '...\n')

## 4. Load Base Model + Fine-Tuned Adapter

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
DRIVE_ADAPTER = '/content/drive/MyDrive/cocos2dx-finetune/cocos2dx-lora-adapter'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

# Load fine-tuned model (base + LoRA)
ft_model = PeftModel.from_pretrained(base_model, DRIVE_ADAPTER)
ft_model.eval()
print('Models loaded.')

## 5. Define Generation Functions

In [ ]:
ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
"""

RAG_PROMPT = """Below is an instruction that describes a task. Use the provided documentation context to write an accurate response.

### Documentation Context:
{context}

### Instruction:
{instruction}

### Response:
"""

def generate(model, prompt, max_new_tokens=512):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def generate_finetuned(instruction):
    prompt = ALPACA_PROMPT.format(instruction=instruction)
    return generate(ft_model, prompt)

def generate_rag(instruction, top_k=5):
    chunks = retrieve_context(instruction, top_k=top_k)
    context = '\n---\n'.join(c['text'] for c in chunks)
    prompt = RAG_PROMPT.format(context=context, instruction=instruction)
    return generate(base_model, prompt)

# Quick sanity check
test_q = test_ds[0]['instruction']
print('Q:', test_q[:150])
print('\n--- Fine-tuned ---')
print(generate_finetuned(test_q)[:300])
print('\n--- RAG ---')
print(generate_rag(test_q)[:300])

## 6. Run Evaluation on Full Test Set

In [ ]:
import json
from tqdm import tqdm

EVAL_RESULTS_PATH = '/content/drive/MyDrive/cocos2dx-finetune/eval_results.json'

# Load partial results if resuming
if os.path.exists(EVAL_RESULTS_PATH):
    with open(EVAL_RESULTS_PATH) as f:
        results = json.load(f)
    print(f'Resuming from {len(results)} completed examples')
else:
    results = []

completed_indices = {r['index'] for r in results}

for i in tqdm(range(len(test_ds)), desc='Evaluating'):
    if i in completed_indices:
        continue
    example = test_ds[i]
    instruction = example['instruction']
    reference = example['output']

    ft_output = generate_finetuned(instruction)
    rag_output = generate_rag(instruction)

    results.append({
        'index': i,
        'instruction': instruction,
        'reference': reference,
        'finetuned': ft_output,
        'rag': rag_output,
    })

    # Save every 10 examples for resume support
    if len(results) % 10 == 0:
        with open(EVAL_RESULTS_PATH, 'w') as f:
            json.dump(results, f)

# Final save
with open(EVAL_RESULTS_PATH, 'w') as f:
    json.dump(results, f)
print(f'Evaluation complete: {len(results)} examples')

## 7. Compute Metrics

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import re

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

def extract_code_blocks(text):
    """Extract code from markdown fenced blocks or treat whole text as code."""
    blocks = re.findall(r'```[\w]*\n(.*?)```', text, re.DOTALL)
    return '\n'.join(blocks) if blocks else text

def extract_identifiers(code):
    """Extract API identifiers like cc.Sprite, node.addChild, etc."""
    return set(re.findall(r'\b(?:cc|ccui|sp|ccs)\.[A-Za-z_.]+\b', code))

def compute_metrics(prediction, reference):
    # Tokenize for BLEU
    ref_tokens = nltk.word_tokenize(reference.lower())
    pred_tokens = nltk.word_tokenize(prediction.lower())

    # BLEU-4
    bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth)

    # ROUGE-L
    rouge = scorer.score(reference, prediction)['rougeL'].fmeasure

    # API identifier recall (CodeBLEU-lite)
    ref_ids = extract_identifiers(reference)
    pred_ids = extract_identifiers(prediction)
    if ref_ids:
        id_recall = len(ref_ids & pred_ids) / len(ref_ids)
        id_precision = len(ref_ids & pred_ids) / len(pred_ids) if pred_ids else 0
        id_f1 = 2 * id_precision * id_recall / (id_precision + id_recall) if (id_precision + id_recall) > 0 else 0
    else:
        id_recall = id_precision = id_f1 = None

    # Code block presence
    has_code = bool(re.search(r'```|\bfunction\b|\bconst\b|\bvar\b|\blet\b|=>|cc\.', prediction))

    return {
        'bleu': bleu,
        'rouge_l': rouge,
        'api_id_recall': id_recall,
        'api_id_f1': id_f1,
        'has_code': has_code,
    }

In [ ]:
# Compute metrics for both models
ft_metrics = []
rag_metrics = []

for r in results:
    ft_m = compute_metrics(r['finetuned'], r['reference'])
    rag_m = compute_metrics(r['rag'], r['reference'])
    ft_metrics.append(ft_m)
    rag_metrics.append(rag_m)

def avg(values):
    valid = [v for v in values if v is not None]
    return sum(valid) / len(valid) if valid else 0

def summarize(metrics, label):
    print(f'\n{"="*50}')
    print(f'  {label}')
    print(f'{"="*50}')
    print(f'  BLEU-4:           {avg([m["bleu"] for m in metrics]):.4f}')
    print(f'  ROUGE-L:          {avg([m["rouge_l"] for m in metrics]):.4f}')
    print(f'  API ID Recall:    {avg([m["api_id_recall"] for m in metrics]):.4f}')
    print(f'  API ID F1:        {avg([m["api_id_f1"] for m in metrics]):.4f}')
    print(f'  Has Code (%):     {avg([1 if m["has_code"] else 0 for m in metrics])*100:.1f}%')

summarize(ft_metrics, 'FINE-TUNED MODEL')
summarize(rag_metrics, 'RAG (Context7 Docs)')

## 8. Per-Example Comparison Table

In [ ]:
import pandas as pd

rows = []
for i, r in enumerate(results):
    rows.append({
        'idx': r['index'],
        'instruction': r['instruction'][:80] + '...',
        'ft_bleu': ft_metrics[i]['bleu'],
        'rag_bleu': rag_metrics[i]['bleu'],
        'ft_rouge': ft_metrics[i]['rouge_l'],
        'rag_rouge': rag_metrics[i]['rouge_l'],
        'ft_api_f1': ft_metrics[i]['api_id_f1'] or 0,
        'rag_api_f1': rag_metrics[i]['api_id_f1'] or 0,
        'winner': 'FT' if ft_metrics[i]['rouge_l'] > rag_metrics[i]['rouge_l'] else 'RAG',
    })

df = pd.DataFrame(rows)
print(f"\nWin rate — FT: {(df['winner']=='FT').mean()*100:.1f}%  RAG: {(df['winner']=='RAG').mean()*100:.1f}%")
df.sort_values('ft_bleu', ascending=False).head(20)

## 9. Side-by-Side Examples

In [ ]:
from IPython.display import display, HTML
import html

def show_comparison(idx):
    r = results[idx]
    ft_m = ft_metrics[idx]
    rag_m = rag_metrics[idx]

    display(HTML(f"""
    <div style='border:1px solid #ccc; padding:12px; margin:8px 0; border-radius:8px'>
    <h3>Example {r['index']}</h3>
    <p><b>Instruction:</b> {html.escape(r['instruction'][:300])}</p>
    <table style='width:100%; border-collapse:collapse'>
    <tr>
      <th style='width:50%; border:1px solid #ddd; padding:8px; background:#e8f5e9'>Fine-Tuned (BLEU={ft_m['bleu']:.3f}, ROUGE={ft_m['rouge_l']:.3f})</th>
      <th style='width:50%; border:1px solid #ddd; padding:8px; background:#e3f2fd'>RAG (BLEU={rag_m['bleu']:.3f}, ROUGE={rag_m['rouge_l']:.3f})</th>
    </tr>
    <tr>
      <td style='border:1px solid #ddd; padding:8px; vertical-align:top'><pre style='white-space:pre-wrap; font-size:11px'>{html.escape(r['finetuned'][:800])}</pre></td>
      <td style='border:1px solid #ddd; padding:8px; vertical-align:top'><pre style='white-space:pre-wrap; font-size:11px'>{html.escape(r['rag'][:800])}</pre></td>
    </tr>
    <tr><td colspan='2' style='border:1px solid #ddd; padding:8px; background:#fff3e0'>
      <b>Reference:</b><pre style='white-space:pre-wrap; font-size:11px'>{html.escape(r['reference'][:500])}</pre>
    </td></tr>
    </table></div>
    """))

# Show 10 diverse examples: 5 where FT wins, 5 where RAG wins
ft_wins = df[df['winner'] == 'FT'].nlargest(5, 'ft_rouge').index.tolist()
rag_wins = df[df['winner'] == 'RAG'].nlargest(5, 'rag_rouge').index.tolist()

print('=== Examples where Fine-Tuned wins ===')
for idx in ft_wins:
    show_comparison(idx)

print('\n=== Examples where RAG wins ===')
for idx in rag_wins:
    show_comparison(idx)

## 10. Summary Chart

In [ ]:
import matplotlib.pyplot as plt

metrics_names = ['BLEU-4', 'ROUGE-L', 'API ID Recall', 'API ID F1', 'Has Code %']
ft_vals = [
    avg([m['bleu'] for m in ft_metrics]),
    avg([m['rouge_l'] for m in ft_metrics]),
    avg([m['api_id_recall'] for m in ft_metrics]),
    avg([m['api_id_f1'] for m in ft_metrics]),
    avg([1 if m['has_code'] else 0 for m in ft_metrics]),
]
rag_vals = [
    avg([m['bleu'] for m in rag_metrics]),
    avg([m['rouge_l'] for m in rag_metrics]),
    avg([m['api_id_recall'] for m in rag_metrics]),
    avg([m['api_id_f1'] for m in rag_metrics]),
    avg([1 if m['has_code'] else 0 for m in rag_metrics]),
]

x = range(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar([i - width/2 for i in x], ft_vals, width, label='Fine-Tuned', color='#4CAF50')
bars2 = ax.bar([i + width/2 for i in x], rag_vals, width, label='RAG (Context7)', color='#2196F3')

ax.set_ylabel('Score')
ax.set_title('Fine-Tuned vs RAG (Context7 Docs) — Test Set Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.set_ylim(0, 1)

for bar in bars1 + bars2:
    h = bar.get_height()
    ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
               xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/cocos2dx-finetune/eval_chart.png', dpi=150)
plt.show()
print('Chart saved to Drive.')

## 11. Save Full Report

In [ ]:
report = {
    'num_examples': len(results),
    'finetuned': {
        'bleu': avg([m['bleu'] for m in ft_metrics]),
        'rouge_l': avg([m['rouge_l'] for m in ft_metrics]),
        'api_id_recall': avg([m['api_id_recall'] for m in ft_metrics]),
        'api_id_f1': avg([m['api_id_f1'] for m in ft_metrics]),
        'has_code_pct': avg([1 if m['has_code'] else 0 for m in ft_metrics]),
    },
    'rag_context7': {
        'bleu': avg([m['bleu'] for m in rag_metrics]),
        'rouge_l': avg([m['rouge_l'] for m in rag_metrics]),
        'api_id_recall': avg([m['api_id_recall'] for m in rag_metrics]),
        'api_id_f1': avg([m['api_id_f1'] for m in rag_metrics]),
        'has_code_pct': avg([1 if m['has_code'] else 0 for m in rag_metrics]),
    },
    'ft_win_rate': float((df['winner'] == 'FT').mean()),
    'rag_win_rate': float((df['winner'] == 'RAG').mean()),
}

report_path = '/content/drive/MyDrive/cocos2dx-finetune/eval_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print(f'\nReport saved to {report_path}')